# router_voto — Laya fina para el router del chat de deepseek-flow

**Qué hace:** fine-tuning completo (encoder incluido) del checkpoint *multilingual* de Laya con
`layaft` (RLCD + calibración de temperatura), sobre casos generados con DeepSeek y pasados por
juez ciego para la pregunta del router de deepseek-flow (¿esta pregunta necesita herramientas o
se responde directa?). Al final mide el test set de 24 casos escritos a mano, jamás entrenados
(la task replica byte a byte la pregunta `PREGUNTA_ROUTER` de producción; el estado es
`{"pregunta": ...}` como en producción).

**Entrada:** el dataset `roquedb/flow-router-voto` (task YAML + contexts + test + datos verificados).
**Acelerador:** GPU T4 (layaft usa una). **Internet:** sí, para pip y para el dataset del base.

**Cómo reproducirlo local:** `train/generate_data.py generate n=70 context=all task=train/tasks/router_voto.yaml`
y `train/generate_data.py verify task=train/tasks/router_voto.yaml` en el repo de bmo.

In [ ]:
!nvidia-smi
%pip install -q laya-finetune==0.1.2 pyyaml
import layaft, laya, torch
print("layaft listo | laya", laya.__version__, "| torch", torch.__version__, "| GPUs:", torch.cuda.device_count())

In [ ]:
import glob, os, shutil, pathlib
W = pathlib.Path("/kaggle/working")
for name in ("tasks/router_voto.yaml", "tasks/contexts_router.txt", "tasks/router_voto_test.jsonl", "data/router_voto_verified.jsonl"):
    hits = glob.glob(f"/kaggle/input/**/{os.path.basename(name)}", recursive=True)
    assert hits, f"no encontré {name}: fijate que el dataset flow-router-voto esté agregado como Input"
    dest = W / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(hits[0], dest)
print("listo:", sorted(str(p.relative_to(W)) for p in W.rglob("*") if p.is_file()))

In [ ]:
# Dónde está el base: el directorio de input que contiene model.safetensors (no confiar en el slug de mount)
import pathlib
candidatos = [p for p in pathlib.Path("/kaggle/input").rglob("model.safetensors")]
print("inputs:", [str(p) for p in pathlib.Path("/kaggle/input").glob("*")])
assert candidatos, "no encontré el checkpoint base: revisá que bmo-laya-base esté como Input"
BASE = str(candidatos[0].parent)
print("base:", BASE)
# Entrenamiento desde el base local; ctx por defecto 1k -> runs/router_voto-1k
!layaft train task=tasks/router_voto.yaml data=data/router_voto_verified.jsonl model={BASE} profile=full

In [ ]:
# Validación en el test set escrito a mano (el número que cuenta)
import pathlib
runs = sorted(p for p in pathlib.Path("runs").glob("*"))
print("runs:", [str(r) for r in runs])
RUN = str(runs[0])
!layaft val task=tasks/router_voto.yaml model={RUN}

In [ ]:
import glob, pathlib
for f in sorted(pathlib.Path("runs/val").rglob("*")):
    if f.is_file():
        print("=" * 20, f, "=" * 20)
        print(f.read_text(encoding="utf-8")[:2000])

## Para traerlo a casa

El checkpoint queda en `runs/router_voto-1k/` (pesos + tokenizer + config con temperatura
calibrada). En la máquina local:

```
kaggle kernels output roquedb/router-voto -p train/runs-kaggle/
```

y en el `.env` de deepseek-flow: `LAYA_MODEL=<ruta>/train/runs-kaggle/runs/router_voto-1k`.